In [21]:
import argparse
import os
import pickle
import time

from importlib import metadata
import torch
try:
    try:
        if metadata.version("rsl-rl"):
            raise ImportError
    except metadata.PackageNotFoundError:
        if metadata.version("rsl-rl-lib") != "3.1.1":  #2.2.4
            raise ImportError
except (metadata.PackageNotFoundError, ImportError) as e:
    raise ImportError("Please uninstall 'rsl_rl' and install 'rsl-rl-lib==2.2.4'.") from e
from rsl_rl.runners import OnPolicyRunner

In [22]:
exec(open('/choreonoid_ws/install/share/irsl_choreonoid/sample/irsl_import.py').read())

In [23]:
from hrp2_env_cnoid import BP000Env as RLEnv

In [24]:
# 任意設定項目
exp_name = 'hrp2-walking2'  # ckpt = 1000
ckpt = 800

action_scale = 1.0 # 動作のスケールを調整

In [25]:
# 既存のセルを置き換え
import pandas as pd
import numpy as np

# データ収集用のリスト
# action_data = []
obs_data = []
torque_data = []
step_data = []
dof_pos_data = []
dof_vel_data = []
# ★新規追加
base_pos_data = []      # ロボット重心位置 (X, Y, Z)
base_vel_data = []      # ロボット実速度 (X, Y, Z)
command_data = []       # 指示速度 (X, Y, Yaw)


# CSVファイルの準備
csv_filename = f'obs_data/{exp_name}_step_data.csv' # # 落ち着かせるためにもう少し待つ（ダンピングで揺れを吸収）
            # print("Settling initial pose...")
            # for _ in range(100):
            #     # 目標値を保持し続ける
            #     self.robot.control_dofs_position(initial_angles, self.dofs_idx_local)
            #     self.scene.step()
os.makedirs('obs_data', exist_ok=True)

In [26]:
def _obs_vec(obs):
    # TensorDict or dict → 'policy' を優先
    if isinstance(obs, dict) or hasattr(obs, "get"):
        if "policy" in obs:
            obs = obs["policy"]
    if torch.is_tensor(obs):
        return obs.detach().cpu().numpy().ravel()
    return np.asarray(obs, dtype=np.float32).ravel()

In [27]:
## set robot path fix collisiton 
ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))  # /userdir
robot_path = os.path.join(os.getcwd(), "hrp2_description/HRP2_genesis.urdf")

In [28]:
log_dir = f"logs/{exp_name}"
env_cfg, obs_cfg, reward_cfg, command_cfg, train_cfg = pickle.load(open(f"logs/{exp_name}/cfgs.pkl", "rb"))
reward_cfg["reward_scales"] = {}

In [29]:
## override
env_cfg["episode_length_s"] = 60.0
# command_cfg["lin_vel_x_range"] = [0.5, 0.5]
env_cfg['dt'] = 0.01
env_cfg['substeps'] = 10
# env_cfg["kd"] = 50
env_cfg['base_roll_noise'] = [0,0]
env_cfg['base_pitch_noise'] = [0,0]
env_cfg['termination_if_roll_greater_than'] = 100
env_cfg['termination_if_pitch_greater_than'] = 100
env_cfg['rotorInertia'] = 1.0
env_cfg["base_init_pos"] = [0.0, 0.0, 0.71]
# command_cfg["ang_vel_range"] = [0.5, 0.5]  # yaw rate

In [30]:
reward_cfg

{'tracking_sigma': 0.25,
 'base_height_target': 0.7,
 'feet_height_target': 0.075,
 'reward_scales': {}}

In [31]:
env_cfg



{'num_actions': 30,
 'default_joint_angles': {'RLEG_JOINT0': 0.0,
  'RLEG_JOINT1': 0.0,
  'RLEG_JOINT2': -0.4,
  'RLEG_JOINT3': 0.8,
  'RLEG_JOINT4': -0.4,
  'RLEG_JOINT5': 0.0,
  'LLEG_JOINT0': 0.0,
  'LLEG_JOINT1': 0.0,
  'LLEG_JOINT2': -0.4,
  'LLEG_JOINT3': 0.8,
  'LLEG_JOINT4': -0.4,
  'LLEG_JOINT5': 0.0,
  'CHEST_JOINT0': 0.0,
  'CHEST_JOINT1': 0.0,
  'HEAD_JOINT0': 0.0,
  'HEAD_JOINT1': 0.0,
  'RARM_JOINT0': 0.0,
  'RARM_JOINT1': -0.1,
  'RARM_JOINT2': 0.0,
  'RARM_JOINT3': -0.6,
  'RARM_JOINT4': 0.0,
  'RARM_JOINT5': 0.0,
  'RARM_JOINT6': 0.0,
  'LARM_JOINT0': 0.0,
  'LARM_JOINT1': 0.1,
  'LARM_JOINT2': 0.0,
  'LARM_JOINT3': -0.6,
  'LARM_JOINT4': 0.0,
  'LARM_JOINT5': 0.0,
  'LARM_JOINT6': 0.0},
 'joint_names': ['RLEG_JOINT0',
  'RLEG_JOINT1',
  'RLEG_JOINT2',
  'RLEG_JOINT3',
  'RLEG_JOINT4',
  'RLEG_JOINT5',
  'LLEG_JOINT0',
  'LLEG_JOINT1',
  'LLEG_JOINT2',
  'LLEG_JOINT3',
  'LLEG_JOINT4',
  'LLEG_JOINT5',
  'CHEST_JOINT0',
  'CHEST_JOINT1',
  'HEAD_JOINT0',
  'HEAD_JOINT1

In [32]:
env = RLEnv(
    num_envs=1,
    env_cfg=env_cfg,
    obs_cfg=obs_cfg,
    reward_cfg=reward_cfg,
    command_cfg=command_cfg,
    dt=env_cfg['dt'],
    substeps=env_cfg['substeps'],
    show_viewer=True,
    robot_urdf_path=robot_path,
)

In [33]:
runner = OnPolicyRunner(env, train_cfg, log_dir, device='cuda')
resume_path = os.path.join(log_dir, f"model_{ckpt}.pt")
runner.load(resume_path)
policy = runner.get_inference_policy(device='cuda')

obs, _ = env.reset()
cnt = 0

torques = env.sim.sbody.getTorques()
dof_pos = env.dof_pos[0].cpu().numpy()
dof_vel = env.dof_vel[0].cpu().numpy()
current_base_pos = env.base_pos[0].cpu().numpy()        # ★追加: [x, y, z]
current_base_vel = np.copy(env.sim.sbody.rootLink.v)
current_commands = env.commands[0].cpu().numpy()        # ★追加: [vx_cmd, vy_cmd, vyaw_cmd]

print("obs : ", obs["policy"])

# データを記録
step_data.append(cnt)
obs_data.append(_obs_vec(obs))
torque_data.append(torques.copy())
dof_pos_data.append(dof_pos)
dof_vel_data.append(dof_vel)
base_pos_data.append(current_base_pos)      # ★追加
base_vel_data.append(current_base_vel)      # ★追加
command_data.append(current_commands)       # ★追加

cnt += 1

--------------------------------------------------------------------------------
Resolved observation sets: 
	 policy :  ['policy']
	 critic :  ['policy']
--------------------------------------------------------------------------------
Actor MLP: MLP(
  (0): Linear(in_features=99, out_features=512, bias=True)
  (1): ELU(alpha=1.0)
  (2): Linear(in_features=512, out_features=256, bias=True)
  (3): ELU(alpha=1.0)
  (4): Linear(in_features=256, out_features=128, bias=True)
  (5): ELU(alpha=1.0)
  (6): Linear(in_features=128, out_features=30, bias=True)
)
Critic MLP: MLP(
  (0): Linear(in_features=99, out_features=512, bias=True)
  (1): ELU(alpha=1.0)
  (2): Linear(in_features=512, out_features=256, bias=True)
  (3): ELU(alpha=1.0)
  (4): Linear(in_features=256, out_features=128, bias=True)
  (5): ELU(alpha=1.0)
  (6): Linear(in_features=128, out_features=1, bias=True)
)
obs :  tensor([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        

In [34]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    dof_pos = env.dof_pos[0].cpu().numpy()
    dof_vel = env.dof_vel[0].cpu().numpy()
    current_base_pos = env.base_pos[0].cpu().numpy()        # ★追加
    current_base_vel = env.sim.sbody.rootLink.v
    current_commands = env.commands[0].cpu().numpy()        # ★追加

    print("torques:", torques)
    print("dof_pos:", dof_pos)
    print("dof_vel:", dof_vel)

    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    dof_pos_data.append(dof_pos)
    dof_vel_data.append(dof_vel)
    base_pos_data.append(current_base_pos)      # ★追加
    base_vel_data.append(current_base_vel)      # ★追加
    command_data.append(current_commands)       # ★追加


    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 1
Original actions :  tensor([[-0.0224, -0.2517,  0.6823, -0.8553, -0.4899, -0.2740,  0.0789,  0.2509,
          0.8128, -0.4818,  0.4087,  0.3216,  0.0036, -0.8995,  0.1958, -0.0896,
          0.2437,  0.4913,  0.3725,  0.2236, -0.0413,  0.1908,  0.0094,  0.1523,
          0.0993,  0.0040,  0.2328, -0.0115, -0.1150,  0.0775]],
       device='cuda:0')
Scaled actions :  tensor([[-0.0224, -0.2517,  0.6823, -0.8553, -0.4899, -0.2740,  0.0789,  0.2509,
          0.8128, -0.4818,  0.4087,  0.3216,  0.0036, -0.8995,  0.1958, -0.0896,
          0.2437,  0.4913,  0.3725,  0.2236, -0.0413,  0.1908,  0.0094,  0.1523,
          0.0993,  0.0040,  0.2328, -0.0115, -0.1150,  0.0775]],
       device='cuda:0')
obs :  tensor([[ 1.9491e-12,  2.5695e-09, -3.4588e-11,  6.5898e-11, -2.1318e-13,
         -1.0000e+00,  1.0000e+00,  0.0000e+00,  0.0000e+00,  7.8404e-13,
          3.8964e-12,  0.0000e+00,  0.0000e+00,  0.0000e+00,  2.3123e-13,
         -6.6849e-13, -4.3343e-12,  0.0000e+00,  0.0000e+00, 

In [35]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    dof_pos = env.dof_pos[0].cpu().numpy()
    dof_vel = env.dof_vel[0].cpu().numpy()
    current_base_pos = env.base_pos[0].cpu().numpy()        # ★追加
    current_base_vel = env.sim.sbody.rootLink.v
    current_commands = env.commands[0].cpu().numpy()        # ★追加

    print("torques:", torques)
    print("dof_pos:", dof_pos)
    print("dof_vel:", dof_vel)

    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    dof_pos_data.append(dof_pos)
    dof_vel_data.append(dof_vel)
    base_pos_data.append(current_base_pos)      # ★追加
    base_vel_data.append(current_base_vel)      # ★追加
    command_data.append(current_commands)       # ★追加


    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 2
Original actions :  tensor([[-0.1605, -0.2716,  0.5977, -1.0109, -0.9829, -0.7685,  0.1788,  0.2447,
          0.8560, -0.2492,  0.5123,  0.5559, -0.1478, -1.1057,  0.5926, -0.0582,
          0.6091,  0.7320,  0.4789,  0.8295, -0.3325,  0.6092,  0.2323,  0.2084,
         -0.0920,  0.0709,  0.7904,  0.2094, -0.1639,  0.2625]],
       device='cuda:0')
Scaled actions :  tensor([[-0.1605, -0.2716,  0.5977, -1.0109, -0.9829, -0.7685,  0.1788,  0.2447,
          0.8560, -0.2492,  0.5123,  0.5559, -0.1478, -1.1057,  0.5926, -0.0582,
          0.6091,  0.7320,  0.4789,  0.8295, -0.3325,  0.6092,  0.2323,  0.2084,
         -0.0920,  0.0709,  0.7904,  0.2094, -0.1639,  0.2625]],
       device='cuda:0')
obs :  tensor([[-6.3460e-03, -2.2139e-02, -1.8182e-02, -3.8430e-04,  1.3698e-04,
         -1.0000e+00,  1.0000e+00,  0.0000e+00,  0.0000e+00, -1.3537e-04,
         -2.3983e-03,  8.4300e-03, -9.9976e-03, -8.0371e-03, -4.8689e-03,
          1.3243e-03,  2.8286e-03,  7.8165e-03, -9.4790e-03, 

In [36]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    dof_pos = env.dof_pos[0].cpu().numpy()
    dof_vel = env.dof_vel[0].cpu().numpy()
    current_base_pos = env.base_pos[0].cpu().numpy()        # ★追加
    current_base_vel = env.sim.sbody.rootLink.v
    current_commands = env.commands[0].cpu().numpy()        # ★追加

    print("torques:", torques)
    print("dof_pos:", dof_pos)
    print("dof_vel:", dof_vel)

    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    dof_pos_data.append(dof_pos)
    dof_vel_data.append(dof_vel)
    base_pos_data.append(current_base_pos)      # ★追加
    base_vel_data.append(current_base_vel)      # ★追加
    command_data.append(current_commands)       # ★追加


    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 3
Original actions :  tensor([[-0.2750, -0.2538,  0.5347, -0.9218, -0.9735, -0.9687,  0.2389,  0.2151,
          0.8048,  0.0442,  0.3604,  0.4024, -0.1111, -1.1597,  0.6798, -0.0224,
          0.7268,  0.7290,  0.4511,  1.0147, -0.4658,  0.7799,  0.3509,  0.1116,
         -0.0535,  0.1509,  0.9540,  0.3032, -0.2427,  0.4122]],
       device='cuda:0')
Scaled actions :  tensor([[-0.2750, -0.2538,  0.5347, -0.9218, -0.9735, -0.9687,  0.2389,  0.2151,
          0.8048,  0.0442,  0.3604,  0.4024, -0.1111, -1.1597,  0.6798, -0.0224,
          0.7268,  0.7290,  0.4511,  1.0147, -0.4658,  0.7799,  0.3509,  0.1116,
         -0.0535,  0.1509,  0.9540,  0.3032, -0.2427,  0.4122]],
       device='cuda:0')
obs :  tensor([[-5.3128e-03, -4.1240e-02,  3.2077e-02, -1.8819e-03,  3.8616e-04,
         -1.0000e+00,  1.0000e+00,  0.0000e+00,  0.0000e+00, -3.4648e-03,
         -8.3676e-03,  2.9768e-02, -3.6895e-02, -3.0146e-02, -1.8281e-02,
          5.6978e-03,  8.9864e-03,  2.7037e-02, -2.5408e-02, 

In [18]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    dof_pos = env.dof_pos[0].cpu().numpy()
    dof_vel = env.dof_vel[0].cpu().numpy()
    current_base_pos = env.base_pos[0].cpu().numpy()        # ★追加
    current_base_vel = env.sim.sbody.rootLink.v
    current_commands = env.commands[0].cpu().numpy()        # ★追加

    print("torques:", torques)
    print("dof_pos:", dof_pos)
    print("dof_vel:", dof_vel)

    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    dof_pos_data.append(dof_pos)
    dof_vel_data.append(dof_vel)
    base_pos_data.append(current_base_pos)      # ★追加
    base_vel_data.append(current_base_vel)      # ★追加
    command_data.append(current_commands)       # ★追加


    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 4
Original actions :  tensor([[-0.2285, -0.2631,  0.6700, -1.0508, -0.8406, -1.0900,  0.2091,  0.2207,
          0.7422,  0.3283,  0.0971,  0.1514, -0.0320, -1.2447,  0.6880, -0.0474,
          0.7921,  0.5571,  0.5027,  0.9874, -0.5610,  0.8389,  0.4271, -0.0368,
          0.0918,  0.1659,  0.9815,  0.3174, -0.2419,  0.4202]],
       device='cuda:0')
Scaled actions :  tensor([[-0.2285, -0.2631,  0.6700, -1.0508, -0.8406, -1.0900,  0.2091,  0.2207,
          0.7422,  0.3283,  0.0971,  0.1514, -0.0320, -1.2447,  0.6880, -0.0474,
          0.7921,  0.5571,  0.5027,  0.9874, -0.5610,  0.8389,  0.4271, -0.0368,
          0.0918,  0.1659,  0.9815,  0.3174, -0.2419,  0.4202]],
       device='cuda:0')
obs :  tensor([[-1.1659e-03, -3.2000e-02,  3.0912e-02, -3.4553e-03,  4.6241e-04,
         -9.9999e-01,  1.0000e+00,  0.0000e+00,  0.0000e+00, -1.1919e-02,
         -1.6277e-02,  5.3010e-02, -6.7856e-02, -6.1983e-02, -4.1303e-02,
          1.2942e-02,  1.6233e-02,  5.0199e-02, -3.3681e-02, 

In [19]:
# 既存のforループを置き換え
num_steps = 500
for i in range(num_steps):
    with torch.no_grad():
        actions = policy(obs)
        
        # アクションスケーリング
        scaled_actions = actions * action_scale
        obs, rews, dones, infos = env.step(scaled_actions)  # スケール済みを使用
        torques = env.sim.sbody.getTorques()
        dof_pos = env.dof_pos[0].cpu().numpy()
        dof_vel = env.dof_vel[0].cpu().numpy()
        current_base_pos = env.base_pos[0].cpu().numpy()        # ★追加
        current_base_vel = np.copy(env.sim.sbody.rootLink.v)
        current_commands = env.commands[0].cpu().numpy()        # ★追加

        # データを記録
        step_data.append(cnt)
        obs_data.append(_obs_vec(obs))
        torque_data.append(torques.copy())
        dof_pos_data.append(dof_pos)
        dof_vel_data.append(dof_vel)
        base_pos_data.append(current_base_pos)      # ★追加
        base_vel_data.append(current_base_vel)      # ★追加
        command_data.append(current_commands)       # ★追加

        # デバッグ表示（最初の数ステップのみ）
        if i < 5:
            print(f"Step {i}: pos={current_base_pos[:2]}, vel={current_base_vel[:2]}")
        
        if i % 20 == 0:
            print(f"Step {i+1}/{num_steps}, Total steps: {cnt}")
            print(f"Position: {current_base_pos[:2]}")
            print(f"Calculated velocity: {current_base_vel[:2]}")
        
        cnt += 1

print(f"データ収集完了: {num_steps} steps collected with action_scale={action_scale}")

Step 0: pos=[ 0.01089592 -0.00126309], vel=[ 0.40216774 -0.04042894]
Step 1/500, Total steps: 5
Position: [ 0.01089592 -0.00126309]
Calculated velocity: [ 0.40216774 -0.04042894]
Step 1: pos=[ 0.01468727 -0.00166805], vel=[ 0.36156501 -0.03864022]
Step 2: pos=[ 0.02131654 -0.00203235], vel=[ 0.65995356 -0.01533786]
Step 3: pos=[ 0.0275431  -0.00195412], vel=[0.59516732 0.02442141]
Step 4: pos=[ 0.0332229  -0.00158314], vel=[0.52359297 0.04839496]
Step 21/500, Total steps: 25
Position: [ 0.11074837 -0.00410035]
Calculated velocity: [ 0.583504   -0.05771638]
Step 41/500, Total steps: 45
Position: [ 0.23601055 -0.00993338]
Calculated velocity: [0.53644194 0.06991666]
Step 61/500, Total steps: 65
Position: [0.3532295  0.01898175]
Calculated velocity: [0.73406978 0.11064882]
Step 81/500, Total steps: 85
Position: [0.49513087 0.04434869]
Calculated velocity: [0.70465601 0.1596072 ]
Step 101/500, Total steps: 105
Position: [0.6381986  0.07819413]
Calculated velocity: [0.85954644 0.34928061]
S

In [20]:
env.sim.stop()

In [ ]:
env.reset()
cnt = 0

torques = env.sim.sbody.getTorques()
dof_pos = env.dof_pos[0].cpu().numpy()
dof_vel = env.dof_vel[0].cpu().numpy()
current_base_pos = env.base_pos[0].cpu().numpy()        # ★追加: [x, y, z]
current_base_vel = env.sim.sbody.rootLink().v
current_commands = env.commands[0].cpu().numpy()        # ★追加: [vx_cmd, vy_cmd, vyaw_cmd]

print("obs : ", obs["policy"])

# データを記録
step_data.append(cnt)
obs_data.append(_obs_vec(obs))
torque_data.append(torques.copy())
dof_pos_data.append(dof_pos)
dof_vel_data.append(dof_vel)
base_pos_data.append(current_base_pos)      # ★追加
base_vel_data.append(current_base_vel)      # ★追加
command_data.append(current_commands)       # ★追加

cnt += 1

In [25]:
# 最もシンプルな保存方法
def save_simple_csv():
    if not step_data:
        print("データがありません")
        return
    
    # 基本的な辞書形式でデータを整理
    data_dict = {'step': step_data}
    
    # # Actionデータ
    # action_array = np.array(action_data)
    # for i in range(action_array.shape[1]):
    #     data_dict[f'action_{i}'] = action_array[:, i]
    
    # Observationデータ
    obs_array = np.array(obs_data)
    for i in range(obs_array.shape[1]):
        data_dict[f'obs_{i}'] = obs_array[:, i]
    
    # Torqueデータ
    torque_array = np.array(torque_data)
    for i in range(torque_array.shape[1]):
        data_dict[f'torque_{i}'] = torque_array[:, i]

    # dof_posデータ
    dof_pos_array = np.array(dof_pos_data)
    for i in range(dof_pos_array.shape[1]):
        data_dict[f'dof_pos_{i}'] = dof_pos_array[:, i]

    # dof_velデータ
    dof_vel_array = np.array(dof_vel_data)
    for i in range(dof_vel_array.shape[1]):
        data_dict[f'dof_vel_{i}'] = dof_vel_array[:, i]

    # ★新規追加: ベース位置 (base_pos)
    if base_pos_data is not None and len(base_pos_data) > 0:
        base_pos_array = np.array(base_pos_data)
        pos_labels = ['base_pos_x', 'base_pos_y', 'base_pos_z']
        for i, label in enumerate(pos_labels):
            data_dict[label] = base_pos_array[:, i]

    # ★新規追加: ベース速度 (base_vel)
    if base_vel_data is not None and len(base_vel_data) > 0:
        base_vel_array = np.array(base_vel_data)
        vel_labels = ['base_vel_x', 'base_vel_y', 'base_vel_z']
        for i, label in enumerate(vel_labels):
            data_dict[label] = base_vel_array[:, i]

    # ★新規追加: 指示速度 (commands)
    if command_data is not None and len(command_data) > 0:
        command_array = np.array(command_data)
        cmd_labels = ['cmd_vel_x', 'cmd_vel_y', 'cmd_vel_yaw']
        for i, label in enumerate(cmd_labels):
            data_dict[label] = command_array[:, i]

    # --- ★追加修正: 長さの確認と調整 ---
    # まず、各データの長さを確認（デバッグ用に出力しても良いです）
    for k, v in data_dict.items():
        print(f"{k}: {len(v)}")
    
    # DataFrameを作成して保存
    df = pd.DataFrame(data_dict)
    csv_filename = f'obs_data_paper/cnoid_{exp_name}_ckpt{ckpt}_scale{action_scale}_paper_1.csv'
    df.to_csv(csv_filename, index=False)
    
    print(f"シンプル版を保存: {csv_filename}")
    print(f"データ形状: {df.shape}")
    
    return df

# シンプル版を実行
df_simple = save_simple_csv()

step: 1001
obs_0: 1001
obs_1: 1001
obs_2: 1001
obs_3: 1001
obs_4: 1001
obs_5: 1001
obs_6: 1001
obs_7: 1001
obs_8: 1001
obs_9: 1001
obs_10: 1001
obs_11: 1001
obs_12: 1001
obs_13: 1001
obs_14: 1001
obs_15: 1001
obs_16: 1001
obs_17: 1001
obs_18: 1001
obs_19: 1001
obs_20: 1001
obs_21: 1001
obs_22: 1001
obs_23: 1001
obs_24: 1001
obs_25: 1001
obs_26: 1001
obs_27: 1001
obs_28: 1001
obs_29: 1001
obs_30: 1001
obs_31: 1001
obs_32: 1001
obs_33: 1001
obs_34: 1001
obs_35: 1001
obs_36: 1001
obs_37: 1001
obs_38: 1001
obs_39: 1001
obs_40: 1001
obs_41: 1001
obs_42: 1001
obs_43: 1001
obs_44: 1001
torque_0: 1001
torque_1: 1001
torque_2: 1001
torque_3: 1001
torque_4: 1001
torque_5: 1001
torque_6: 1001
torque_7: 1001
torque_8: 1001
torque_9: 1001
torque_10: 1001
torque_11: 1001
dof_pos_0: 1001
dof_pos_1: 1001
dof_pos_2: 1001
dof_pos_3: 1001
dof_pos_4: 1001
dof_pos_5: 1001
dof_pos_6: 1001
dof_pos_7: 1001
dof_pos_8: 1001
dof_pos_9: 1001
dof_pos_10: 1001
dof_pos_11: 1001
dof_vel_0: 1001
dof_vel_1: 1001
dof_ve

In [51]:
cam_coords = ib.cameraPositionLookingAt([10.0, 0.0, 5.0], [1, 0, 0], [1, 0, 1])

In [21]:
ib.getCameraCoordsParam()

{'pos': [9.960047598067828, 0.18383649520435325, 6.53269958710866],
 'aa': [-0.673939338297673,
  -0.6630079594020114,
  0.3259236322583489,
  2.5017586127773153],
 'fov': 0.6108652381980153}

In [28]:
cam_coords = ib.cameraPositionLookingAt([0.0, 0.0, 18.0], [0, 0, 0], [1, 0, 0])
cam_coords

<coordinates[0x589b91a0fce0] 0 0 18 / 0.707107 -0.707107 -4.32978e-17 4.32978e-17 >

In [29]:
ib.setCameraCoords(cam_coords)

In [50]:
cam_coords = ib.cameraPositionLookingAt([8.0, 0, 00], [0, 0, 0], [0, 0, 0])
cam_coords

<coordinates[0x589bab4dde80] 8 0 0 / -4.32978e-17 -0.707107 -4.32978e-17 0.707107 >

In [52]:
ib.setCameraCoords(cam_coords)

In [ ]:
"RLEG_JOINT0": 0.0,   # R_HIP_Y
"RLEG_JOINT1": 0.0,   # R_HIP_R
"RLEG_JOINT2": -0.4,  # R_HIP_P
"RLEG_JOINT3": 0.8,   # R_KNEE_P
"RLEG_JOINT4": -0.4,  # R_ANKLE_P
"RLEG_JOINT5": 0.0,   # R_ANKLE_R
"LLEG_JOINT0": 0.0,   # L_HIP_Y
"LLEG_JOINT1": 0.0,   # L_HIP_R
"LLEG_JOINT2": -0.4,  # L_HIP_P
"LLEG_JOINT3": 0.8,   # L_KNEE_P
"LLEG_JOINT4": -0.4,  # L_ANKLE_P
"LLEG_JOINT5": 0.0,   # L_ANKLE_R